In [9]:
!pip install langchain langchain-core langchain-groq langsmith python-dotenv -q

In [16]:
import os

os.environ["GROQ_API_KEY"] = "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"

In [17]:
import os
print(os.environ["GROQ_API_KEY"])

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


In [18]:
from groq import Groq
import os

client = Groq(api_key=os.environ["GROQ_API_KEY"])

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say OK in one word"}]
)

print(response.choices[0].message.content)

Okay


In [56]:
from langchain_core.prompts import PromptTemplate

extraction_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
You are a resume parser.

Resume:
{resume}

Return ONLY valid JSON:

{{
  "name": "",
  "years_experience": 0,
  "education": "",
  "skills": [],
  "tools": [],
  "key_projects": []
}}

IMPORTANT:
Return ONLY JSON.
No explanation.
No markdown.
"""
)

In [57]:
matching_prompt = PromptTemplate(
    input_variables=["extracted_profile", "job_description"],
    template="""
Job:
{job_description}

Profile:
{extracted_profile}

Return ONLY JSON:

{{
  "matched_skills": [],
  "missing_skills": [],
  "experience_match": "",
  "education_match": "",
  "overall_match": ""
}}

IMPORTANT:
Return ONLY JSON.
No explanation.
No markdown.
"""
)

In [58]:
scoring_prompt = PromptTemplate(
    input_variables=["extracted_profile", "match_analysis", "job_description"],
    template="""
Job:
{job_description}

Profile:
{extracted_profile}

Match:
{match_analysis}

Return ONLY JSON:

{{
  "skills_score": 0,
  "experience_score": 0,
  "education_score": 0,
  "tools_score": 0,
  "total_score": 0,
  "grade": "",
  "recommendation": "",
  "explanation": ""
}}

IMPORTANT:
Return ONLY JSON.
No explanation.
No markdown.
"""
)

In [59]:
extraction_chain = extraction_prompt | get_llm(0.0)
matching_chain = matching_prompt | get_llm(0.0)
scoring_chain = scoring_prompt | get_llm(0.1)

In [60]:
JOB_DESCRIPTION = """Data Scientist role ..."""

RESUME_STRONG = """Priya Sharma 4 years ..."""
RESUME_AVERAGE = """Rahul Verma 2.5 years ..."""
RESUME_WEAK = """Anil Kumar 6 months ..."""

In [61]:
import json
import re

def safe_json(text):
    try:
        text = re.sub(r"```json|```", "", text).strip()

        start = text.find("{")
        end = text.rfind("}") + 1

        if start != -1 and end != -1:
            text = text[start:end]

        return json.loads(text)

    except:
        return {"raw": text}

In [62]:
def safe_llm_call(chain, input_dict):
    try:
        return chain.invoke(input_dict).content
    except Exception as e:
        return f"ERROR: {e}"

In [63]:
def screen_resume(resume, job, label):

    print("\n====================")
    print("Candidate:", label)
    print("====================")

    # Step 1
    r1 = safe_llm_call(extraction_chain, {"resume": resume})
    profile = safe_json(r1)

    # Step 2
    r2 = safe_llm_call(matching_chain, {
        "extracted_profile": r1,
        "job_description": job
    })
    match = safe_json(r2)

    # Step 3
    r3 = safe_llm_call(scoring_chain, {
        "extracted_profile": r1,
        "match_analysis": r2,
        "job_description": job
    })

    print("\n🔴 RAW SCORE OUTPUT:\n", r3)

    score = safe_json(r3)

    print("\nScore:", score.get("total_score", "N/A"))
    print("Recommendation:", score.get("recommendation", "N/A"))

    return {
        "profile": profile,
        "match": match,
        "score": score
    }

In [64]:
result1 = screen_resume(RESUME_STRONG, JOB_DESCRIPTION, "STRONG")
result2 = screen_resume(RESUME_AVERAGE, JOB_DESCRIPTION, "AVERAGE")
result3 = screen_resume(RESUME_WEAK, JOB_DESCRIPTION, "WEAK")


Candidate: STRONG

🔴 RAW SCORE OUTPUT:
 {
  "skills_score": 0,
  "experience_score": 0,
  "education_score": 0,
  "tools_score": 0,
  "total_score": 0,
  "grade": "F",
  "recommendation": "Acquire necessary skills and experience",
  "explanation": "Lack of required skills, incomplete education information, and insufficient experience"
}

Score: 0
Recommendation: Acquire necessary skills and experience

Candidate: AVERAGE

🔴 RAW SCORE OUTPUT:
 {
  "skills_score": 0,
  "experience_score": 0,
  "education_score": 0,
  "tools_score": 0,
  "total_score": 0,
  "grade": "F",
  "recommendation": "Improve skills and gain more experience",
  "explanation": "Candidate lacks required skills and experience for the Data Scientist role"
}

Score: 0
Recommendation: Improve skills and gain more experience

Candidate: WEAK

🔴 RAW SCORE OUTPUT:
 {
  "skills_score": 0,
  "experience_score": 0,
  "education_score": 0,
  "tools_score": 0,
  "total_score": 0,
  "grade": "F",
  "recommendation": "Gain more e